In [ ]:
#v1.3 BI-GRU
import os
import glob
import gc
import joblib
import math
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error, r2_score
from sklearn.neighbors import NearestNeighbors
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

PROJECT_PATH = "/content/drive/MyDrive/StockPrediction_2024_FINAL"
STOCK_DIR = os.path.join(PROJECT_PATH, "data_final", "stocks")
SENTIMENT_PATH = os.path.join(PROJECT_PATH, "data_final", "unified_sentiment_layer_CLAMPED.csv")

MODEL_SAVE_DIR = os.path.join(PROJECT_PATH, "saved_models_v1_f")
PLOT_SAVE_DIR = os.path.join(PROJECT_PATH, "validation_plots_v1_f")
LOSS_PLOT_DIR = os.path.join(PROJECT_PATH, "loss_plots_v1_f")
HISTORY_SAVE_DIR = os.path.join(PROJECT_PATH, "training_history_v1_f")
RESULTS_PATH = os.path.join(PROJECT_PATH, "v1_f_metrics.csv")

for d in [MODEL_SAVE_DIR, PLOT_SAVE_DIR, LOSS_PLOT_DIR, HISTORY_SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

stock_files = glob.glob(os.path.join(STOCK_DIR, "*.csv"))
print("Assets detected:", len(stock_files))

df_sentiment = pd.read_csv(SENTIMENT_PATH, parse_dates=['Date'])


def profit_focused_loss(y_true, y_pred):
    direction_true = tf.sign(y_true)
    direction_pred = tf.sign(y_pred)
    direction_error = tf.cast(tf.not_equal(direction_true, direction_pred), tf.float32)
    magnitude_error = tf.abs(y_true - y_pred)
    move_size = tf.abs(y_true)
    size_weight = tf.where(move_size > 0.01, 2.0, 1.0)
    loss = (direction_error * 5.0 + 1.0) * magnitude_error * size_weight
    return tf.reduce_mean(loss)

def build_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Bidirectional(GRU(64, return_sequences=True,
                          kernel_regularizer=tf.keras.regularizers.l2(0.001))),
        Dropout(0.25),
        GRU(32, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        Dense(1)
    ])
    model.compile(optimizer='nadam', loss=profit_focused_loss)
    return model

def create_sequences(X, y, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
        ys.append(y.iloc[i])
    return np.array(Xs), np.array(ys)

def transductive_adjustment(model, X_train, y_train, X_test):
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)

    knn = NearestNeighbors(n_neighbors=20, metric='cosine')
    knn.fit(X_train_flat)

    base_preds = model.predict(X_test, verbose=0).flatten()
    _, indices = knn.kneighbors(X_test_flat)

    enhanced = []
    for i, neighbor_idx in enumerate(indices):
        pattern_bias = np.mean(y_train[neighbor_idx])
        enhanced_pred = 0.7 * base_preds[i] + 0.3 * pattern_bias
        enhanced.append(enhanced_pred)

    return np.array(enhanced)

# Training Loop

results = []
N_ENSEMBLE = 3
LOOKBACK = 15

print("\nStarting V1.3 training...\n")

for idx, file_path in enumerate(stock_files):

    ticker = os.path.basename(file_path).replace(".csv", "")
    print(f"[{idx+1}/{len(stock_files)}] {ticker}")

    try:
        df = pd.read_csv(file_path, index_col=0, parse_dates=True)

        sent = df_sentiment[df_sentiment['Ticker'] == ticker]
        df = df.merge(sent, on='Date', how='left')


        df.fillna(0, inplace=True)
        df = df.infer_objects(copy=False)

        df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
        df['Target_Log_Returns'] = df['Log_Returns'].shift(-1)
        df.dropna(inplace=True)

        if len(df) < 200:
            print("   Skipped (too few rows)")
            continue

        features = [
            'Close','Log_Returns','RSI','SMA_50',
            'Volatility_20D','USD_INR',
            'Macro_Sentiment_7D','Stock_Sentiment_7D',
            'Sentiment_Momentum','News_Shock'
        ]

        split = int(len(df) * 0.85)
        train = df.iloc[:split]
        test = df.iloc[split:]

        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(train[features])
        X_test_scaled = scaler.transform(test[features])

        joblib.dump(scaler, os.path.join(MODEL_SAVE_DIR, f"{ticker}_scaler.pkl"))

        X_train, y_train = create_sequences(X_train_scaled, train['Target_Log_Returns'], LOOKBACK)
        X_test, y_test = create_sequences(X_test_scaled, test['Target_Log_Returns'], LOOKBACK)

        preds = []
        histories = []

        for ens in range(N_ENSEMBLE):
            tf.keras.backend.clear_session()
            model = build_model((X_train.shape[1], X_train.shape[2]))

            early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

            history = model.fit(
                X_train, y_train,
                epochs=25,
                batch_size=32,
                validation_split=0.2,
                verbose=0,
                callbacks=[early_stop]
            )

            histories.append(history.history)
            model.save(os.path.join(MODEL_SAVE_DIR, f"{ticker}_ensemble_{ens}.keras"))

            pred = transductive_adjustment(model, X_train, y_train, X_test)
            preds.append(pred)

        avg_pred = np.mean(preds, axis=0)

        plt.figure(figsize=(8,5))
        for i, hist in enumerate(histories):
            plt.plot(hist['loss'], label=f'Train {i}')
            plt.plot(hist['val_loss'], linestyle='--', label=f'Val {i}')
        plt.title(f"{ticker} Loss Curve")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(LOSS_PLOT_DIR, f"{ticker}_loss.png"))
        plt.close()

        hist_df = pd.DataFrame(histories[0])
        hist_df.to_csv(os.path.join(HISTORY_SAVE_DIR, f"{ticker}_history.csv"), index=False)

        base_prices = test['Close'].iloc[LOOKBACK:].values
        min_len = min(len(base_prices), len(avg_pred))

        actual = base_prices[:min_len] * np.exp(y_test[:min_len])
        predicted = base_prices[:min_len] * np.exp(avg_pred[:min_len])

        mape = mean_absolute_percentage_error(actual, predicted) * 100
        rmse = math.sqrt(mean_squared_error(actual, predicted))
        mae = mean_absolute_error(actual, predicted)
        actual_var = np.var(actual)
        mean_actual = np.mean(actual)
        cv = np.std(actual) / mean_actual if mean_actual != 0 else 0

        if actual_var < 1e-4 or cv < 0.001:
            r2 = np.nan
        else:
            r2 = r2_score(actual, predicted)

            if r2 < -1.0:
                r2 = np.nan

        results.append({
            'Ticker': ticker,
            'MAPE': mape,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2
        })

        plt.figure(figsize=(10,5))
        plt.plot(actual[-80:], label='Actual')
        plt.plot(predicted[-80:], linestyle='--', label='Predicted')
        plt.title(f"{ticker} | MAPE {mape:.2f}%")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_SAVE_DIR, f"{ticker}_validation.png"))
        plt.close()

        print(f"   ✅ MAPE: {mape:.2f}% | R²: {r2 if not np.isnan(r2) else 'N/A (Low Variance)'}")
        gc.collect()

    except Exception as e:
        print("   ❌ Error:", e)


if results:
    df_res = pd.DataFrame(results)
    df_res.to_csv(RESULTS_PATH, index=False)


    avg_mape = df_res['MAPE'].dropna().mean()
    avg_rmse = df_res['RMSE'].dropna().mean()
    avg_mae  = df_res['MAE'].dropna().mean()
    avg_r2   = df_res['R2'].dropna().mean()

    print("\n============================================================")
    print("V1.3 PIPELINE COMPLETED (FINAL METRICS)")
    print("============================================================")
    print(f"Assets Processed : {len(df_res)}")
    print(f"Average MAPE     : {avg_mape:.4f} %")
    print(f"Average RMSE     : {avg_rmse:.4f}")
    print(f"Average MAE      : {avg_mae:.4f}")
    print(f"Average R²       : {avg_r2:.4f}")
    print("============================================================")

Mounted at /content/drive
Assets detected: 58

Starting V1.3 training...

[1/58] RELIANCE.NS
   ✅ MAPE: 0.98% | R²: 0.9716231318282094
[2/58] TCS.NS
   ✅ MAPE: 0.96% | R²: 0.9890991162529877
[3/58] HDFCBANK.NS
   ✅ MAPE: 0.79% | R²: 0.9813125377610576
[4/58] INFY.NS
   ✅ MAPE: 1.13% | R²: 0.9738421077062185
[5/58] ICICIBANK.NS
   ✅ MAPE: 0.83% | R²: 0.9701635446606642
[6/58] HINDUNILVR.NS
   ✅ MAPE: 0.90% | R²: 0.9680542326924914
[7/58] SBIN.NS
   ✅ MAPE: 1.02% | R²: 0.9868226466374379
[8/58] BHARTIARTL.NS
   ✅ MAPE: 1.01% | R²: 0.9858200287907584
[9/58] KOTAKBANK.NS
   ✅ MAPE: 0.93% | R²: 0.9704784595457251
[10/58] ITC.NS
   ✅ MAPE: 0.85% | R²: 0.9805095708722854
[11/58] LTIM.NS
   ✅ MAPE: 1.31% | R²: 0.9681778059527241
[12/58] HCLTECH.NS
   ✅ MAPE: 1.09% | R²: 0.9615260523231011
[13/58] AXISBANK.NS
   ✅ MAPE: 0.98% | R²: 0.9690377748333774
[14/58] LT.NS
   ✅ MAPE: 0.99% | R²: 0.9628242669129644
[15/58] ASIANPAINT.NS
   ✅ MAPE: 0.99% | R²: 0.9871388146838539
[16/58] MARUTI.NS
   ✅ MAP

/tmp/ipykernel_202/3322129404.py:129: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(0, inplace=True)


   ✅ MAPE: 0.96% | R²: 0.9921400945611029
[43/58] BANKBEES.NS


/tmp/ipykernel_202/3322129404.py:129: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(0, inplace=True)


   ✅ MAPE: 0.61% | R²: 0.9862677944166243
[44/58] NIFTYBEES.NS


...

   ✅ MAPE: 0.66% | R²: 0.4207801833815491
[57/58] CPSEETF.NS


/tmp/ipykernel_202/3322129404.py:129: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(0, inplace=True)


   ✅ MAPE: 0.94% | R²: 0.9628196488658015
[58/58] HAVELLS.NS


/tmp/ipykernel_202/3322129404.py:129: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(0, inplace=True)


   ✅ MAPE: 1.34% | R²: 0.9171053394432225

🚀 V1.3 PIPELINE COMPLETED (FINAL METRICS)
Assets Processed : 58
Average MAPE     : 1.0658 %
Average RMSE     : 30.4097
Average MAE      : 22.2916
Average R²       : 0.9582
